# 🔴 LIVE Alpaca Trading — REAL MONEY

**This notebook trades the LIVE Alpaca account. Every order submitted here moves real money.**

Rules of this notebook:
- **Manual only.** It is NOT wired into `run_all.py`, launchd, the watchdog, or any automation. It runs only when you run it, cell by cell.
- **How it relates to the pipeline.** The scheduled pipeline (`run_all.py --trade` → `paper_trade.py`) now trades this same LIVE account automatically. This notebook is the manual counterpart: same signal CSVs, same planning logic, same `Reports/live_orders_log.csv` — but it runs only when you run it, cell by cell, and nothing is submitted until you arm it.
- **Plan first, then arm.** The plan cell is a dry run. Nothing is submitted until you set `LIVE_ARMED = True` and run the execution cell.
- **Separate keys.** Uses `ALPACA_LIVE_KEY_ID` / `ALPACA_LIVE_SECRET_KEY` from `.env` — never the paper keys. Key values are never printed.
- **Same proven logic.** Order planning reuses `paper_trade.py` (`plan_orders`, `apply_buying_power_guard`, `check_signal_freshness`): whole shares, SELL-first, buy-signal-only, buying-power cap, fail-closed on stale signals.
- **Order types mirror the automated pipeline.** Market open → DAY market orders. Market closed (before 7 PM CT) → DAY limit orders at the close with extended hours. After 7 PM CT the execution cell refuses — run it in the morning instead.

In [ ]:
# ── Cell 1: setup ──────────────────────────────────────────────────────
import os, sys, math, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
from dotenv import load_dotenv

ROOT = os.path.expanduser("~/Desktop/Stock Analysis")
os.chdir(ROOT)
sys.path.insert(0, ROOT)
load_dotenv(os.path.join(ROOT, ".env"))

# ⛔ Safety switch: the execution cell refuses to run unless you flip this to True
#    AFTER reviewing the plan.
LIVE_ARMED = False

LIVE_KEY_VAR, LIVE_SECRET_VAR = "ALPACA_LIVE_KEY_ID", "ALPACA_LIVE_SECRET_KEY"
LIVE_ORDER_LOG = os.path.join(ROOT, "Reports", "live_orders_log.csv")  # never touches paper_orders_log.csv
CT = ZoneInfo("America/Chicago")


def live_trading_client():
    """Alpaca TradingClient for the LIVE account (paper=False). Key names only; values are never printed."""
    from alpaca.trading.client import TradingClient
    key, secret = os.getenv(LIVE_KEY_VAR), os.getenv(LIVE_SECRET_VAR)
    if not key or not secret:
        raise SystemExit(
            f"Missing live keys: add {LIVE_KEY_VAR} and {LIVE_SECRET_VAR} to {ROOT}/.env "
            "(your LIVE Alpaca API key pair — NOT the paper keys). Nothing was submitted.")
    return TradingClient(key, secret, paper=False)


print("setup ok | LIVE_ARMED =", LIVE_ARMED)
print("live keys present:", bool(os.getenv(LIVE_KEY_VAR)), bool(os.getenv(LIVE_SECRET_VAR)))
print("live order log:", LIVE_ORDER_LOG)

In [ ]:
# ── Cell 2: read-only LIVE account snapshot (no orders) ─────────────────
client = live_trading_client()
acct = client.get_account()
equity, bp, cash = float(acct.equity), float(acct.buying_power), float(acct.cash)
live_positions = client.get_all_positions()
positions = {p.symbol: float(p.qty) for p in live_positions}

print(f"LIVE account  equity ${equity:,.2f} | buying power ${bp:,.2f} | cash ${cash:,.2f}")
print(f"positions: {len(positions)}")
if live_positions:
    display(pd.DataFrame([{"Symbol": p.symbol, "Qty": float(p.qty),
                           "MarketValue": float(p.market_value),
                           "AvgEntry": float(p.avg_entry_price),
                           "UnrealizedPL": float(p.unrealized_pl)} for p in live_positions]))
else:
    print("(no positions)")

In [ ]:
# ── Cell 3: PLAN — dry run, nothing is submitted ────────────────────────
from paper_trade import plan_orders, apply_buying_power_guard, check_signal_freshness

client = live_trading_client()
as_of = check_signal_freshness()  # fail-closed: raises on stale/missing signals
clock = client.get_clock()
if not clock.is_open:
    print("⚠️ Market is currently CLOSED — the execution cell will submit extended-hours DAY limit "
          "orders at the close price (only before 7 PM CT); after 7 PM it refuses — run it in the morning.")

acct = client.get_account()
equity, bp, cash = float(acct.equity), float(acct.buying_power), float(acct.cash)
positions = {p.symbol: float(p.qty) for p in client.get_all_positions()}

# Same proven planner as the automated pipeline: whole shares, SELL-first,
# buy-signal-only, HOLD rows never traded.
orders, meta, targets = plan_orders(source="auto", account_size=equity,
                                    positions=positions, fractional=False)
orders = apply_buying_power_guard(orders, bp)  # BUY spend capped at live buying power

trade = orders[orders["Side"].isin(["SELL", "BUY"])].copy()
print(f"target source: {meta['source']} | signal as of: {meta['as_of']} | strategy: {meta['strategy']}")
print(f"LIVE equity ${equity:,.2f} | buying power ${bp:,.2f} | cash ${cash:,.2f}")
if trade.empty:
    print("\nNo orders — everything is HOLD/SKIP. Nothing would be submitted.")
else:
    print(f"\n{len(trade)} orders PLANNED (NOT submitted):")
    display(trade[["Symbol", "Side", "Shares", "Price", "Est_Value"]])
    est_buy = float(trade.loc[trade["Side"] == "BUY", "Est_Value"].sum())
    print(f"estimated BUY spend: ${est_buy:,.2f} (capped at buying power ${bp:,.2f})")

## ⛔ Stop and review

The table above is the **entire** live order plan. If it looks right and you want to execute:

1. Set `LIVE_ARMED = True` in Cell 1 and re-run Cell 1.
2. Run the execution cell below.

**When to run it:** after the Mon/Wed/Fri 15:15 CT pipeline run (fresh signals), or on a trading morning during regular hours.
The execution cell **re-verifies** signal freshness and buying power, then **re-plans** from the latest data before submitting — so what it sends always reflects the newest verified state, not a stale preview.
Order types: market open → DAY market orders; market closed (before 7 PM CT) → DAY limit orders at the close with extended hours (these can go partially unfilled — check the VERIFY cell); after 7 PM → it refuses.

In [ ]:
# ── Cell 5: EXECUTION — submits REAL orders ────────────────────────────
if not LIVE_ARMED:
    raise SystemExit("⛔ LIVE_ARMED is False — nothing was submitted. "
                     "Review the plan, set LIVE_ARMED = True in Cell 1, re-run Cell 1, then run this cell.")

from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from paper_trade import plan_orders, apply_buying_power_guard, check_signal_freshness

# Re-verify everything at execution time (fail closed).
as_of = check_signal_freshness()  # also proves today had a completed bar (a trading day)
client = live_trading_client()
now_ct = datetime.now(CT)
if client.get_clock().is_open:
    session = "regular"   # regular hours -> DAY market orders
elif now_ct.hour < 19:
    session = "extended"  # post-close, before 7 PM CT -> DAY limit orders at the close, extended_hours=True
else:
    raise SystemExit("Extended hours are over (past 7:00 PM CT) — nothing was submitted. "
                     "Run this notebook during regular market hours tomorrow morning.")
acl = client.get_account()
equity, bp = float(acl.equity), float(acl.buying_power)
positions = {p.symbol: float(p.qty) for p in client.get_all_positions()}

orders, meta, targets = plan_orders(source="auto", account_size=equity,
                                    positions=positions, fractional=False)
orders = apply_buying_power_guard(orders, bp)
trade_rows = orders[orders["Side"].isin(["SELL", "BUY"])]

if trade_rows.empty:
    print("Nothing to trade (all HOLD/SKIP). No orders submitted.")
else:
    date_str = now_ct.strftime("%Y%m%d")
    results = []
    for r in trade_rows.itertuples():  # SELL rows come first (plan_orders orders SELL before BUY)
        qty = math.floor(float(r.Shares or 0))
        if qty <= 0:
            results.append((r.Symbol, r.Side, qty, "-", "SKIPPED: zero quantity"))
            continue
        if r.Side == "SELL":
            held = positions.get(r.Symbol, 0)
            if held <= 0:
                results.append((r.Symbol, r.Side, qty, "-", "SKIPPED: not held"))
                continue
            qty = min(qty, math.floor(held))  # never sell more than held
            if qty <= 0:
                results.append((r.Symbol, r.Side, 0, "-", "SKIPPED: nothing to sell"))
                continue
        cid = "live_" + hashlib.sha256(f"LIVE-{date_str}-{session}-{r.Side}-{r.Symbol}-{qty}".encode()).hexdigest()[:24]
        side = OrderSide.SELL if r.Side == "SELL" else OrderSide.BUY
        if session == "regular":
            req = MarketOrderRequest(symbol=r.Symbol, qty=qty, time_in_force=TimeInForce.DAY,
                                     side=side, client_order_id=cid)
            otype = "MKT"
        else:
            px = round(float(r.Price), 2)
            if not px > 0:
                results.append((r.Symbol, r.Side, qty, "-", "SKIPPED: no price"))
                continue
            req = LimitOrderRequest(symbol=r.Symbol, qty=qty, limit_price=px,
                                    time_in_force=TimeInForce.DAY, extended_hours=True,
                                    side=side, client_order_id=cid)
            otype = f"LMT {px:.2f} XH"
        try:
            o = client.submit_order(req)
            results.append((r.Symbol, r.Side, qty, otype, f"SUBMITTED: {o.status} (id {o.id})"))
        except Exception as e:  # keep going; report every failure
            results.append((r.Symbol, r.Side, qty, otype, f"FAILED: {e}"))
    res = pd.DataFrame(results, columns=["Symbol", "Side", "Shares", "Type", "Status"])
    res.insert(0, "TimestampCT", now_ct.strftime("%Y-%m-%d %H:%M:%S"))
    res.insert(1, "Signal_As_Of", as_of)
    res.insert(2, "Session", session)
    res.to_csv(LIVE_ORDER_LOG, mode="a", header=not os.path.exists(LIVE_ORDER_LOG), index=False)
    print(res.to_string(index=False))
    print(f"\nlogged to {LIVE_ORDER_LOG}")
    if session == "extended":
        print("\nNote: extended-hours limit orders can go partially unfilled or unfilled — "
              "check the VERIFY cell and your Alpaca dashboard.")

In [ ]:
# ── Cell 6: VERIFY — live positions after the run ───────────────────────
client = live_trading_client()
after = {p.symbol: float(p.qty) for p in client.get_all_positions()}
tw = targets.set_index("Symbol")["Weight"] if len(targets) else {}
rows = [{"Symbol": s, "LiveQty": q, "TargetWeight": round(float(tw.get(s, 0)), 4)} for s, q in sorted(after.items())]
display(pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Symbol", "LiveQty", "TargetWeight"]))
print("TargetWeight is the fraction of equity each name should hold (0 = not a target).")